# Этап 1 (вертикальный срез) — выгрузка станционных данных (rp5.ru)

Вторая часть «вертикального среза» (см. `PROJECT_BRIEF.md`, раздел 5):

1. автоматически скачивает срочные наблюдения станции **Кандалакша (ВМО 22217)**
   с **rp5.ru** за контрольный период из `config.yaml`;
2. кэширует сырой `.csv.gz` в `data/raw/` (без удаления файла повторный запуск
   не дёргает сеть — раздел 3.1 брифа);
3. парсит CSV, приводит колонки к именам схемы `weather_data` (раздел 3.4
   брифа), помечает `source = "station"`;
4. строит отчёт о покрытии периода и доле наблюдений с кодом текущей погоды
   (WW) — это прямой ответ на п.5 раздела 5 брифа;
5. сохраняет результат в `data/clean/`.

**О источнике.** В отличие от ERA5 (модельный реанализ, см. `01_era5_ingest.ipynb`),
здесь — **реальные срочные наблюдения** метеостанции, обычно раз в 3 часа,
с кодом **текущей погоды WW** (гололёд, метель, мокрый снег и т.п.) — именно
ради этих кодов станция выбрана основным источником (раздел 3.1 брифа). Минус —
возможны пропуски в ряду.

**Важно (методическая пометка):** в этом ноутбуке `WW` сохраняется как сырой
текст (`ww_raw`) — rp5.ru отдаёт его в виде русскоязычного описания явления, а
не числового кода ВМО. Перевод в числовой `ww_code` и булевы флаги явлений
(`icing_flag` и т.п.) — задача Этапа 3 (`process/`), требующая отдельного
словаря соответствий "фраза rp5 → код ВМО".


In [1]:
# Стандартные и сторонние библиотеки.
# requests  - HTTP-запросы (с сессией) к rp5.ru
# re        - извлечение ссылки на файл из ответа сервера
# gzip      - распаковка скачанного .csv.gz "на лету"
# pandas    - таблицы данных
# yaml      - чтение config.yaml
# pathlib   - кроссплатформенная работа с путями
import gzip
import re
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
import yaml


def find_project_root(start: Path) -> Path:
    """Находит корень проекта — каталог, где лежит config.yaml.

    Нужно потому, что рабочий каталог Jupyter зависит от того, как запущен
    ноутбук (из корня проекта или из ingest/).
    """
    for parent in [start, *start.parents]:
        if (parent / "config.yaml").exists():
            return parent
    raise FileNotFoundError("Не найден config.yaml — проверьте, откуда запущен ноутбук")


PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

station = config["station"]
period = config["period"]
station_cfg = config["sources"]["station"]

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Станция: {station['name']} (ВМО {station_cfg['wmo_id']})")
print(f"Период выгрузки: {period['start_date']} — {period['end_date']}")


Корень проекта: /Users/golochalov/IdeaProjects/Climate analysis
Станция: Кандалакша (ВМО 22217)
Период выгрузки: 2024-01-01 — 2024-12-31


## Механика автоматической выгрузки с rp5.ru

У rp5.ru нет официального API, но есть внутренний эндпоинт, которым пользуется
сама страница архива при нажатии кнопки "Выбрать в файл GZ". Алгоритм:

1. `GET` страницы архива станции (`https://rp5.ru/Архив_погоды_в_Кандалакше`)
   через `requests.Session()` — сервер выдаёт сессионную куку `PHPSESSID`.
2. `POST` той же сессией на `https://rp5.ru/responses/reFileSynop.php` с
   параметрами: `wmo_id`, период (`a_date1`/`a_date2` в формате `dd.mm.YYYY`),
   набор кодов дополнительных полей (`f_ed3/f_ed4/f_ed5`), формат периода
   (`f_pe`/`f_pe1`), язык (`lng_id`). В ответ приходит фрагмент `<script>` со
   ссылкой на готовый `.csv.gz`.
3. `GET` по этой ссылке — скачивание самого файла.

Подход проверен вручную перед написанием ноутбука (на январе 2024 — вернул
корректный `.csv.gz` с 29 колонками, включая `WW`).

**Если rp5.ru изменит механизм** (другой эндпоинт/параметры, капча) — ячейка
ниже выведет понятную инструкцию для ручной выгрузки через браузер вместо
падения с непонятной ошибкой.


In [2]:
RAW_DIR = PROJECT_ROOT / config["paths"]["raw_dir"]
RAW_DIR.mkdir(parents=True, exist_ok=True)

wmo_id = station_cfg["wmo_id"]
# rp5.ru ожидает даты в формате dd.mm.YYYY
date1 = pd.Timestamp(period["start_date"]).strftime("%d.%m.%Y")
date2 = pd.Timestamp(period["end_date"]).strftime("%d.%m.%Y")

raw_filename = f"station_kandalaksha_{wmo_id}_{period['start_date']}_{period['end_date']}.csv.gz"
raw_path = RAW_DIR / raw_filename

ARCHIVE_PAGE = "https://rp5.ru/Архив_погоды_в_Кандалакше"
# Кириллица в URL должна быть процентно закодирована перед использованием в
# HTTP-запросе и заголовках (requests/urllib3 требуют latin-1-совместимые
# значения) - для печати человеку (ручной фолбэк) используем ARCHIVE_PAGE как есть.
ARCHIVE_PAGE_URL = quote(ARCHIVE_PAGE, safe=":/")
DOWNLOAD_ENDPOINT = "https://rp5.ru/responses/reFileSynop.php"
USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

if raw_path.exists():
    # Файл уже скачан ранее - сетевые запросы не выполняем
    print(f"Используем кэш: {raw_path.relative_to(PROJECT_ROOT)}")
else:
    print("Кэш не найден, выполняем автоматическую выгрузку с rp5.ru...")
    try:
        session = requests.Session()
        session.headers.update({"User-Agent": USER_AGENT, "Referer": "https://rp5.ru/"})

        # Шаг 1: сессионная кука с главной страницы архива станции
        resp = session.get(ARCHIVE_PAGE_URL, timeout=30)
        resp.raise_for_status()

        # Шаг 2: запрос на генерацию файла за нужный период
        post_data = {
            "wmo_id": wmo_id,
            "a_date1": date1,
            "a_date2": date2,
            "f_ed3": 5,
            "f_ed4": 12,
            "f_ed5": 25,
            "f_pe": 1,
            "f_pe1": 2,
            "lng_id": 2,
        }
        resp = session.post(
            DOWNLOAD_ENDPOINT,
            data=post_data,
            headers={"X-Requested-With": "XMLHttpRequest", "Referer": ARCHIVE_PAGE_URL},
            timeout=60,
        )
        resp.raise_for_status()

        # Ссылка на готовый файл приходит внутри <script>: ...href=URL download>...
        match = re.search(r"href=(https?://\S*?\.csv\.gz)\s+download", resp.text)
        if not match:
            raise RuntimeError(f"Не найдена ссылка на файл в ответе rp5.ru: {resp.text[:300]!r}")
        file_url = match.group(1)

        # Шаг 3: скачивание самого файла
        resp = session.get(file_url, timeout=60)
        resp.raise_for_status()
        raw_path.write_bytes(resp.content)
        print(f"Сохранено: {raw_path.relative_to(PROJECT_ROOT)} ({len(resp.content)} байт)")

    except Exception as exc:
        print(f"Автоматическая выгрузка не удалась: {exc}")
        print()
        print("РУЧНОЙ ФОЛБЭК:")
        print(f"  1. Откройте {ARCHIVE_PAGE}")
        print(f"  2. Задайте период {date1} — {date2}")
        print('  3. Нажмите "Выбрать в файл GZ" и скачайте архив')
        print(f"  4. Сохраните файл как: {raw_path}")
        print("  5. Перезапустите эту ячейку")
        raise


Используем кэш: data/raw/station_kandalaksha_22217_2024-01-01_2024-12-31.csv.gz


## Формат файла и парсинг

Файл — `.csv.gz`, кодировка UTF-8, разделитель `;`, первые 6 строк — служебные
комментарии (название станции, период, ссылка на расшифровку параметров).
Имена колонок и значения обёрнуты в кавычки. Записи идут от **новых дат к
старым** — потребуется сортировка по возрастанию.

**Особенность файла:** каждая строка данных заканчивается на `;` (после
последней колонки `sss` идёт ещё один `;` перед переводом строки), т.е. в
строках данных на одно поле больше, чем имён колонок в заголовке. Без
`index_col=False` pandas в этом случае молча использует первую колонку
(дату/время) как индекс DataFrame и сдвигает все остальные значения на одну
позицию — поэтому передаём `index_col=False` явно.


In [3]:
with gzip.open(raw_path, "rt", encoding="utf-8") as f:
    df_raw = pd.read_csv(f, sep=";", skiprows=6, index_col=False)

# Имена колонок приходят в кавычках - убираем их
df_raw.columns = [c.strip('"') for c in df_raw.columns]

print(f"Колонок: {len(df_raw.columns)}")
print(list(df_raw.columns))

df_raw["datetime"] = pd.to_datetime(df_raw["Местное время в Кандалакше"], format="%d.%m.%Y %H:%M")
df_raw = df_raw.sort_values("datetime").reset_index(drop=True)

print(f"\nСтрок: {len(df_raw)}")
print(f"Период: {df_raw['datetime'].min()} — {df_raw['datetime'].max()}")
df_raw.head()


Колонок: 29
['Местное время в Кандалакше', 'T', 'Po', 'P', 'Pa', 'U', 'DD', 'Ff', 'ff10', 'ff3', 'N', 'WW', 'W1', 'W2', 'Tn', 'Tx', 'Cl', 'Nh', 'H', 'Cm', 'Ch', 'VV', 'Td', 'RRR', 'tR', 'E', 'Tg', "E'", 'sss']

Строк: 2928
Период: 2024-01-01 00:00:00 — 2024-12-31 21:00:00


,Местное время в Кандалакше,T,Po,P,Pa,U,DD,Ff,ff10,ff3,...,Ch,VV,Td,RRR,tR,E,Tg,E',sss,datetime
0,01.01.2024 00:00,-18.1,768.4,771.1,0.9,86,"Ветер, дующий с севера",1,2.0,NaN,...,Перистые (часто в виде полос) и перисто-слоист...,10.0,-19.9,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01 00:00:00
1,01.01.2024 03:00,-16.4,768.7,771.3,0.3,88,"Ветер, дующий с востока",1,1.0,NaN,...,NaN,10.0,-17.9,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01 03:00:00
2,01.01.2024 06:00,-15.6,768.8,771.4,0.1,88,"Ветер, дующий с северо-северо-запада",1,2.0,NaN,...,NaN,10.0,-17.2,Осадков нет,12.0,NaN,NaN,NaN,NaN,2024-01-01 06:00:00
3,01.01.2024 09:00,-14.2,768.6,771.1,-0.2,87,"Ветер, дующий с севера",2,4.0,NaN,...,NaN,10.0,-15.9,Осадков нет,12.0,NaN,NaN,Ровный слой сухого рассыпчатого снега покрывае...,38.0,2024-01-01 09:00:00
4,01.01.2024 12:00,-14.9,768.7,771.4,0.1,89,"Ветер, дующий с севера",1,2.0,NaN,...,Перистые (часто в виде полос) и перисто-слоист...,10.0,-16.3,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01 12:00:00


## Проверка типов колонок

Перед переименованием убеждаемся, что числовые поля (`T`, `U`, `Ff` и т.д.)
действительно числовые, а текстовые (`DD`, `WW` и т.д.) — текстовые: это
быстрая проверка, что парсинг CSV не "съехал" по колонкам.


In [4]:
check_cols = ["T", "U", "Ff", "ff10", "P", "Po", "RRR", "DD", "WW"]
df_raw[check_cols].dtypes


T       float64
U         int64
Ff        int64
ff10    float64
P       float64
Po      float64
RRR      object
DD       object
WW       object
dtype: object

## Приведение к колонкам схемы `weather_data`

Соответствие колонок rp5 → схема (раздел 3.4 брифа), с методическими
пометками:

| rp5    | weather_data       | Примечание |
|--------|---------------------|------------|
| `T`    | `temperature`       | °C, как есть |
| `U`    | `humidity`          | %, как есть |
| `Ff`   | `wind_speed`        | м/с, как есть |
| `ff10` | `wind_gust`         | м/с; передаётся не на каждом сроке (только при значимых порывах за 10 мин до срока) — много NaN, это нормально |
| `P`    | `pressure`          | мм рт. ст. → гПа (`× 1.33322`); взято `P` (на уровне моря), а не `Po`, для сопоставимости с `pressure_msl` ERA5 |
| `RRR`  | `precipitation`     | мм; текстовые значения `"Осадков нет"` и `"Следы осадков"` приводятся к `0.0` (см. ниже), настоящие пропуски остаются `NaN` |
| `WW`   | `ww_raw` (доп. поле)| сырой текст; перевод в числовой `ww_code` ВМО — Этап 3 |

**Про `RRR` отдельно:** в сыром файле это поле смешанное — помимо чисел (мм)
там встречаются текстовые значения `"Осадков нет"` (осадков не зафиксировано)
и `"Следы осадков"` (количество ниже порога измерения, < 0.05 мм). Оба случая
методически означают "осадков практически нет" и приводятся к `0.0`, чтобы
итоговая колонка `precipitation` не оказалась почти полностью `NaN` (это была
бы большая часть строк). Настоящие пропуски (пустое значение в исходном файле)
остаются `NaN`. Различение "осадков нет" и "следы" в отдельный признак —
при необходимости задача Этапа 3.

`snowfall`, `precipitation_type`, `ww_code`, `icing_flag` в этом ноутбуке **не
заполняются** — явный задел для `process/` (Этап 3).


In [5]:
# RRR: текстовые маркеры -> 0.0, остальное -> число, настоящие пропуски -> NaN
rrr_text_to_zero = {"Осадков нет": 0.0, "Следы осадков": 0.0}
rrr_numeric = pd.to_numeric(df_raw["RRR"].replace(rrr_text_to_zero), errors="coerce")

print("RRR: всего строк -", len(df_raw))
print("  числовых значений в исходном файле:", pd.to_numeric(df_raw['RRR'], errors='coerce').notna().sum())
print("  'Осадков нет' / 'Следы осадков' -> 0.0:", df_raw['RRR'].isin(rrr_text_to_zero).sum())
print("  настоящих пропусков (NaN):", rrr_numeric.isna().sum())


RRR: всего строк - 2928
  числовых значений в исходном файле: 574
  'Осадков нет' / 'Следы осадков' -> 0.0: 889
  настоящих пропусков (NaN): 1465


In [6]:
MMHG_TO_HPA = 1.33322

df_station = pd.DataFrame({
    "datetime": df_raw["datetime"],
    "temperature": pd.to_numeric(df_raw["T"], errors="coerce"),
    "humidity": pd.to_numeric(df_raw["U"], errors="coerce"),
    "wind_speed": pd.to_numeric(df_raw["Ff"], errors="coerce"),
    "wind_gust": pd.to_numeric(df_raw["ff10"], errors="coerce"),
    "pressure": pd.to_numeric(df_raw["P"], errors="coerce") * MMHG_TO_HPA,
    "precipitation": rrr_numeric,
    "ww_raw": df_raw["WW"],
    "source": "station",
})

print(f"Строк: {len(df_station)}")
df_station.head(10)


Строк: 2928


,datetime,temperature,humidity,wind_speed,wind_gust,pressure,precipitation,ww_raw,source
0,2024-01-01 00:00:00,-18.1,86,1,2.0,1028.045942,NaN,,station
1,2024-01-01 03:00:00,-16.4,88,1,1.0,1028.312586,NaN,,station
2,2024-01-01 06:00:00,-15.6,88,1,2.0,1028.445908,0.0,,station
3,2024-01-01 09:00:00,-14.2,87,2,4.0,1028.045942,0.0,,station
4,2024-01-01 12:00:00,-14.9,89,1,2.0,1028.445908,NaN,Облака в целом рассеиваются или становятся мен...,station
5,2024-01-01 15:00:00,-14.7,87,1,3.0,1028.579230,NaN,Состояние неба в общем не изменилось.,station
6,2024-01-01 18:00:00,-15.3,88,1,2.0,1028.579230,0.0,,station
7,2024-01-01 21:00:00,-14.4,83,1,3.0,1028.845874,0.0,,station
8,2024-01-02 00:00:00,-18.2,88,1,2.0,1028.845874,NaN,,station
9,2024-01-02 03:00:00,-20.9,85,1,3.0,1028.712552,NaN,,station


## Отчёт о покрытии (раздел 5, п.5 брифа)

- сколько сроков наблюдений ожидалось (каждые 3 часа) и сколько получено
  фактически — пропуски в самом ряде станции;
- доля сроков с непустым `ww_raw` (наблюдённое явление погоды), помесячно —
  это оценка, насколько полно станция отдаёт коды явлений за контрольный год;
- сравнение временного диапазона станции с диапазоном ERA5
  (`01_era5_ingest.ipynb`) — стыковка источников по времени.


In [7]:
# Пропуски в самом ряде станции: срочные наблюдения ожидаются каждые 3 часа
expected_times = pd.date_range(
    start=df_station["datetime"].min().normalize(),
    end=df_station["datetime"].max(),
    freq="3h",
)
missing_times = expected_times.difference(df_station["datetime"])

print(f"Период станции: {df_station['datetime'].min()} — {df_station['datetime'].max()}")
print(f"Ожидается сроков (каждые 3ч): {len(expected_times)}")
print(f"Получено сроков:              {len(df_station)}")
print(f"Пропущено сроков:             {len(missing_times)} "
      f"({len(missing_times) / len(expected_times):.1%})")


Период станции: 2024-01-01 00:00:00 — 2024-12-31 21:00:00
Ожидается сроков (каждые 3ч): 2928
Получено сроков:              2928
Пропущено сроков:             0 (0.0%)


In [8]:
# Доля сроков с непустым WW (наблюдённое явление погоды) по месяцам.
# fillna("") + strip - в файле "пустой" WW бывает либо NaN, либо строкой из пробела.
ww_present = df_station["ww_raw"].fillna("").astype(str).str.strip() != ""

coverage_by_month = (
    pd.DataFrame({"ww_present": ww_present, "month": df_station["datetime"].dt.to_period("M")})
    .groupby("month")["ww_present"]
    .mean()
)

print(f"Всего сроков с непустым WW: {ww_present.sum()} из {len(df_station)} "
      f"({ww_present.mean():.1%})")
print("\nДоля сроков с непустым WW по месяцам:")
coverage_by_month


Всего сроков с непустым WW: 1070 из 2928 (36.5%)

Доля сроков с непустым WW по месяцам:


month
2024-01    0.439516
2024-02    0.577586
2024-03    0.358871
2024-04    0.512500
2024-05    0.266129
2024-06    0.354167
2024-07    0.270161
2024-08    0.149194
2024-09    0.295833
2024-10    0.314516
2024-11    0.379167
2024-12    0.483871
Freq: M, Name: ww_present, dtype: float64

In [9]:
# Сравнение временного диапазона со станцией ERA5 (data/clean/ из ноутбука 1)
CLEAN_DIR = PROJECT_ROOT / config["paths"]["clean_dir"]
era5_path = CLEAN_DIR / f"era5_kandalaksha_{period['start_date']}_{period['end_date']}.csv"

if era5_path.exists():
    df_era5 = pd.read_csv(era5_path, parse_dates=["datetime"])
    print(f"ERA5:    {df_era5['datetime'].min()} — {df_era5['datetime'].max()}, {len(df_era5)} строк (почасово)")
    print(f"Станция: {df_station['datetime'].min()} — {df_station['datetime'].max()}, {len(df_station)} строк (раз в 3ч)")
else:
    print(f"Файл ERA5 не найден ({era5_path.relative_to(PROJECT_ROOT)}) - "
          "сначала выполните 01_era5_ingest.ipynb")


ERA5:    2024-01-01 00:00:00 — 2024-12-31 23:00:00, 8784 строк (почасово)
Станция: 2024-01-01 00:00:00 — 2024-12-31 21:00:00, 2928 строк (раз в 3ч)


## Сохранение результата

Таблица с пометкой `source = "station"` сохраняется в `data/clean/` —
используется в следующем ноутбуке (`db/`) для загрузки в SQLite вместе с ERA5.


In [10]:
clean_path = CLEAN_DIR / f"station_kandalaksha_{period['start_date']}_{period['end_date']}.csv"
df_station.to_csv(clean_path, index=False)
print(f"Сохранено: {clean_path.relative_to(PROJECT_ROOT)} ({len(df_station)} строк)")
df_station.head()


Сохранено: data/clean/station_kandalaksha_2024-01-01_2024-12-31.csv (2928 строк)


,datetime,temperature,humidity,wind_speed,wind_gust,pressure,precipitation,ww_raw,source
0,2024-01-01 00:00:00,-18.1,86,1,2.0,1028.045942,NaN,,station
1,2024-01-01 03:00:00,-16.4,88,1,1.0,1028.312586,NaN,,station
2,2024-01-01 06:00:00,-15.6,88,1,2.0,1028.445908,0.0,,station
3,2024-01-01 09:00:00,-14.2,87,2,4.0,1028.045942,0.0,,station
4,2024-01-01 12:00:00,-14.9,89,1,2.0,1028.445908,NaN,Облака в целом рассеиваются или становятся мен...,station


## Итог

- Срочные наблюдения станции Кандалакша за контрольный период скачаны
  автоматически и закэшированы в `data/raw/`.
- Таблица с пометкой `source = "station"` и сырым текстом WW (`ww_raw`)
  сохранена в `data/clean/`.
- Получен отчёт о пропусках в ряде станции и о доле наблюдений с кодом
  текущей погоды (WW) по месяцам.

**Отложено на Этап 3 (`process/`):**
- перевод `ww_raw` в числовой `ww_code` ВМО и булевы флаги явлений
  (гололёд, мокрый снег и т.п.);
- заполнение `snowfall` / `precipitation_type` (из ERA5 или прокси-критериев).

**Следующий шаг:** загрузка обоих источников (`data/clean/era5_*.csv` и
`data/clean/station_*.csv`) в SQLite по схеме `weather_data` и контрольный
график температуры по обоим источникам — ноутбук `db/01_load_to_sqlite.ipynb`
(пп. 3-4 раздела 5 брифа).
